# 02 — Comparaison dictionnaire nettoyé vs non nettoyé

**Objectif :** mesurer l'impact du nettoyage du dictionnaire sur MEDCON.

- `merged_mesh_snomed_dico.json` — dictionnaire brut (445 186 entrées)
- `cleaned_mesh_snomed_dico.json` — dictionnaire nettoyé (2 515 entrées)




## 0. Setup

In [5]:
import os

if not os.path.exists('/content/Evaluating_medical_machine_translation'):
    !git clone https://github.com/11Maroua/Evaluating_medical_machine_translation.git

os.chdir('/content/Evaluating_medical_machine_translation')
print(f'Répertoire : {os.getcwd()}')

Cloning into 'Evaluating_medical_machine_translation'...
remote: Enumerating objects: 55, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 55 (delta 15), reused 51 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (55/55), 13.14 MiB | 4.69 MiB/s, done.
Resolving deltas: 100% (15/15), done.
Répertoire : /content/Evaluating_medical_machine_translation


In [6]:
from google.colab import files

os.makedirs('data', exist_ok=True)
os.makedirs('results', exist_ok=True)

# Uploader : merged_mesh_snomed_dico.json + cleaned_mesh_snomed_dico.json + corpus_WMT24.json
uploaded = files.upload()
for fname in uploaded:
    dest = f'data/{fname}'
    os.replace(fname, dest)
    print(f'  → {dest}')

Saving cleaned_mesh_snomed_dico.json to cleaned_mesh_snomed_dico.json
Saving corpus_WMT24.json to corpus_WMT24.json
Saving merged_mesh_snomed_dico.json to merged_mesh_snomed_dico.json
  → data/cleaned_mesh_snomed_dico.json
  → data/corpus_WMT24.json
  → data/merged_mesh_snomed_dico.json


In [10]:
!pip install -q pyahocorasick sacrebleu unbabel-comet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.4/101.4 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 84.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.7/529.7 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 1. Imports

In [11]:
import sys
import ahocorasick
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from collections import Counter
from scipy.stats import mannwhitneyu, wilcoxon

os.chdir('/content/Evaluating_medical_machine_translation')
sys.path.insert(0, 'src')
from medcon import build_pairs, build_automaton, medcon_grouped

print('Imports OK')

Imports OK


## 2. Chargement des deux dictionnaires + corpus

In [12]:
with open('data/cleaned_mesh_snomed_dico.json', encoding='utf-8') as f:
    raw_dict_cleaned = json.load(f)

with open('data/merged_mesh_snomed_dico.json', encoding='utf-8') as f:
    raw_dict_merged = json.load(f)

with open('data/corpus_WMT24.json', encoding='utf-8') as f:
    test_docs = json.load(f)

print(f'Dictionnaire nettoyé : {len(raw_dict_cleaned):>10,} entrées EN')
print(f'Dictionnaire brut    : {len(raw_dict_merged):>10,} entrées EN')
print(f'Réduction            : {100*(1 - len(raw_dict_cleaned)/len(raw_dict_merged)):.1f}%')
print(f'Corpus WMT24         : {len(test_docs)} documents')

Dictionnaire nettoyé :      2,515 entrées EN
Dictionnaire brut    :    445,186 entrées EN
Réduction            : 99.4%
Corpus WMT24         : 49 documents


## 3. Construction des automates pour les deux dictionnaires

In [14]:
print('Construction automates — dictionnaire nettoyé...')
pairs_cleaned  = build_pairs(raw_dict_cleaned)
auto_en_cl     = build_automaton(pairs_cleaned, 'en')
auto_fr_cl     = build_automaton(pairs_cleaned, 'fr')
print(f'  Paires : {len(pairs_cleaned):,}  |  Termes EN : {len(auto_en_cl):,}  |  Termes FR : {len(auto_fr_cl):,}')

print('\nConstruction automates — dictionnaire brut ...')
pairs_merged   = build_pairs(raw_dict_merged)
auto_en_mg     = build_automaton(pairs_merged, 'en')
auto_fr_mg     = build_automaton(pairs_merged, 'fr')
print(f'  Paires : {len(pairs_merged):,}  |  Termes EN : {len(auto_en_mg):,}  |  Termes FR : {len(auto_fr_mg):,}')

Construction automates — dictionnaire nettoyé...
  Paires : 2,515  |  Termes EN : 2,515  |  Termes FR : 3,186

Construction automates — dictionnaire brut ...
  Paires : 445,186  |  Termes EN : 440,065  |  Termes FR : 357,615


## 4. MEDCON sur les deux dictionnaires

In [15]:
def run_medcon(test_docs, pairs, auto_en, auto_fr, label):
    print(f'\nMEDCON — {label}')
    results = []
    for i, doc in enumerate(tqdm(test_docs, desc=label)):
        r = medcon_grouped(
            doc['text_en'],
            doc['gpt_translation'],
            pairs, auto_en, auto_fr
        )
        results.append({
            'doc_id'           : i,
            'medcon_f1'        : r['f1'],
            'medcon_precision' : r['precision'],
            'medcon_recall'    : r['recall'],
            'n_expected'       : r['n_expected'],
            'n_found'          : r['n_found'],
            'n_match'          : r['n_match'],
            'en_concepts'      : r['en_concepts'],
            'matched'          : r['matched'],
            'missed'           : r['missed'],
            'extra'            : r['extra'],
        })
    df = pd.DataFrame(results)
    # Docs avec au moins 1 paire attendue (les seuls interprétables)
    df_valid = df[df['n_expected'] > 0]
    print(f'  Docs avec ≥1 paire attendue : {len(df_valid)}/{len(df)}')
    print(f'  Paires attendues (moy.)     : {df["n_expected"].mean():.2f}')
    print(f'  Precision                   : {df_valid["medcon_precision"].mean():.3f}')
    print(f'  Recall                      : {df_valid["medcon_recall"].mean():.3f}')
    print(f'  F1                          : {df_valid["medcon_f1"].mean():.3f}')
    return df


df_cleaned = run_medcon(test_docs, pairs_cleaned, auto_en_cl, auto_fr_cl, 'Nettoyé')
df_merged  = run_medcon(test_docs, pairs_merged,  auto_en_mg, auto_fr_mg, 'Brut')


MEDCON — Nettoyé


Nettoyé:   0%|          | 0/49 [00:00<?, ?it/s]

  Docs avec ≥1 paire attendue : 25/49
  Paires attendues (moy.)     : 0.82
  Precision                   : 0.820
  Recall                      : 0.760
  F1                          : 0.780

MEDCON — Brut


Brut:   0%|          | 0/49 [00:00<?, ?it/s]

  Docs avec ≥1 paire attendue : 49/49
  Paires attendues (moy.)     : 38.43
  Precision                   : 0.494
  Recall                      : 0.450
  F1                          : 0.463


## 5. Tableau comparatif

In [16]:
print('=' * 70)
print('COMPARAISON MEDCON — NETTOYÉ vs BRUT')
print('=' * 70)

for label, df in [('Nettoyé', df_cleaned), ('Brut', df_merged)]:
    df_valid = df[df['n_expected'] > 0]
    print(f"\n  {label}")
    print(f"  {'─'*40}")
    print(f"  Docs avec ≥1 paire attendue : {len(df_valid):>5} / {len(df)}")
    print(f"  Paires attendues (moy.)     : {df['n_expected'].mean():>8.2f}")
    print(f"  Paires trouvées  (moy.)     : {df['n_found'].mean():>8.2f}")
    print(f"  Paires matchées  (moy.)     : {df['n_match'].mean():>8.2f}")
    print(f"  Precision (sur docs valides): {df_valid['medcon_precision'].mean():>8.3f}")
    print(f"  Recall    (sur docs valides): {df_valid['medcon_recall'].mean():>8.3f}")
    print(f"  F1        (sur docs valides): {df_valid['medcon_f1'].mean():>8.3f}")

# Test statistique de Wilcoxon sur les F1 (paired)
print('\n--- Test statistique (Wilcoxon signé-rangé sur F1) ---')
try:
    stat, p = wilcoxon(df_cleaned['medcon_f1'], df_merged['medcon_f1'])
    print(f'  W={stat:.1f}  p={p:.4f}  -> {"différence significative" if p < 0.05 else "pas de différence significative"} (α=0.05)')
except Exception as e:
    print(f'  Test non applicable : {e}')

COMPARAISON MEDCON — NETTOYÉ vs BRUT

  Nettoyé
  ────────────────────────────────────────
  Docs avec ≥1 paire attendue :    25 / 49
  Paires attendues (moy.)     :     0.82
  Paires trouvées  (moy.)     :     0.82
  Paires matchées  (moy.)     :     0.63
  Precision (sur docs valides):    0.820
  Recall    (sur docs valides):    0.760
  F1        (sur docs valides):    0.780

  Brut
  ────────────────────────────────────────
  Docs avec ≥1 paire attendue :    49 / 49
  Paires attendues (moy.)     :    38.43
  Paires trouvées  (moy.)     :    34.41
  Paires matchées  (moy.)     :    16.80
  Precision (sur docs valides):    0.494
  Recall    (sur docs valides):    0.450
  F1        (sur docs valides):    0.463

--- Test statistique (Wilcoxon signé-rangé sur F1) ---
  W=576.0  p=0.7165  -> pas de différence significative (α=0.05)


## 6. Analyse qualitative : termes détectés en plus par le dictionnaire brut

In [17]:
print('TERMES SUPPLÉMENTAIRES DÉTECTÉS PAR LE DICTIONNAIRE BRUT\n')
print(f"{'Doc':>4}  {'Nettoyé':>8}  {'Brut':>8}  Termes EN supplémentaires")
print('-' * 70)

for i in range(len(test_docs)):
    concepts_cl = set(df_cleaned.iloc[i]['en_concepts'])
    concepts_mg = set(df_merged.iloc[i]['en_concepts'])
    extra = concepts_mg - concepts_cl
    n_cl = len(concepts_cl)
    n_mg = len(concepts_mg)
    if extra:
        extra_str = ', '.join(sorted(extra)[:6])
        if len(extra) > 6:
            extra_str += f' ... (+{len(extra)-6})'
        print(f'  {i:>3}  {n_cl:>8}  {n_mg:>8}  {extra_str}')

# Termes supplémentaires les plus fréquents
all_extra_terms = []
for i in range(len(test_docs)):
    concepts_cl = set(df_cleaned.iloc[i]['en_concepts'])
    concepts_mg = set(df_merged.iloc[i]['en_concepts'])
    all_extra_terms.extend(concepts_mg - concepts_cl)

print(f'\nTOP 20 termes EN détectés par le brut mais pas le nettoyé :')
for term, count in Counter(all_extra_terms).most_common(20):
    print(f'  [{count:2d}x] {term}')

TERMES SUPPLÉMENTAIRES DÉTECTÉS PAR LE DICTIONNAIRE BRUT

 Doc   Nettoyé      Brut  Termes EN supplémentaires
----------------------------------------------------------------------
    0         0        40  100, 11, 2, 3, 4, 6 ... (+34)
    1         0        47  11, 2, 5, 6, 7, 8 ... (+41)
    2         0        27  adolescence, adolescent, associated with, bedroom, birth, birth cohort ... (+21)
    3         0        30  adjustment, altered, application, can, clinical trials, depression ... (+24)
    4         0        38  action, active, call, clinical trial, conduct, controlled ... (+32)
    5         0        10  french, health, history, history of, long, only ... (+4)
    6         2        32  10, 11, 7, analysis, anaplasma, animal ... (+24)
    7         1        41  approved, cell, early, eye, failure, gene ... (+34)
    8         0        37  activities, activity, adoption, application, appointment, assessment ... (+31)
    9         4        75  10, 11, 12, 2, 3, 4 ... (+65

## 7. Analyse du bruit : faux positifs dans le dictionnaire nettoyé

In [18]:
print('FAUX POSITIFS (extras) — NETTOYÉ vs BRUT\n')

for label, df in [('Nettoyé', df_cleaned), ('Brut', df_merged)]:
    all_extra = []
    for el in df['extra']:
        all_extra.extend(el)
    counter = Counter(all_extra)
    print(f'{label} — TOP 15 paires extras (terme FR dans traduction mais EN absent source) :')
    for i, (lbl, count) in enumerate(counter.most_common(15), 1):
        print(f'  {i:2d}. [{count:2d}x] {lbl}')
    print(f'  Total extras uniques : {len(counter)}\n')

FAUX POSITIFS (extras) — NETTOYÉ vs BRUT

Nettoyé — TOP 15 paires extras (terme FR dans traduction mais EN absent source) :
   1. [ 3x] aftercare -> post-cure
   2. [ 2x] sadness -> tristesse
   3. [ 1x] diagnostic imaging -> imagerie diagnostique
   4. [ 1x] maternal health -> santé maternelle
   5. [ 1x] acquired immunodeficiency syndrome -> sida
   6. [ 1x] anesthetists -> anesthésistes
  Total extras uniques : 6

Brut — TOP 15 paires extras (terme FR dans traduction mais EN absent source) :
   1. [16x] nursing -> soins
   2. [12x] ratios -> rapport
   3. [12x] ratio -> rapport
   4. [10x] as - dosing instruction fragment -> comme
   5. [ 9x] disease caused by 2019-ncov -> covid-19
   6. [ 8x] therapy (regime/therapy) -> thérapie
   7. [ 8x] bran - chemical -> son
   8. [ 8x] bran -> son
   9. [ 7x] following -> après
  10. [ 7x] therapy -> intervention thérapeutique
  11. [ 7x] health -> santé
  12. [ 6x] objective observation -> objectif
  13. [ 6x] case - situation -> cas
  14. [